In [83]:
using LinearAlgebra

In [84]:
struct Reflector{T}
    v::Vector{T}
    beta::T
end

function reflector(x)
    sign_x1 = x[1] >= 0 #True se positivo, False, se não
    norm_x = norm(x)

    v = copy(x)
    v[1] -= norm_x

    if sign_x1
        beta = (norm_x + x[1])/(norm_x * sum(abs2, @view x[2:end])) 
    else
        beta = 1/(norm_x*(norm_x - x[1]))
    end

    return Reflector(v, beta)
end

reflector (generic function with 1 method)

e) Estabilidade. Teste sua função com vetores x tanto Float64 quanto Float32, e com
pare os resultados.
Dica: verifique que a resposta está sendo automaticamente calculada no tipo certo. Se você
passar um vetor Float32 e receber um Float64 de resposta, alguma conta está “promovendo
tipos”. Uma dica é não ter constantes não-inteiras como 1.0, que são interpretadas como Float64.

f) Escreva uma função calc_beta(v) que calcula β unicamente a partir de v. Repita
os testes Float64 e Float32 usando esta função, e compare os resultados com a função
anterior. Observe que não há muita diferença ao calcular β a partir de x ou apenas de v,
mesmo com x quase alinhado com e1.

In [21]:
function calc_beta(v)   
    v_dot = sum(abs2, v)
    return 2/v_dot
end

calc_beta (generic function with 1 method)

Questão 2. Forma de Hessemberg.
Se você se sentir à vontade, pode definir um tipo de dado para os refletores, e então as funções
desta questão poderiam receber e retornar Refletores, ao invés de v e β separadamente.
a) Aplicando refletores. Escreva uma função apply_reflector(v, beta, b) que cal
cula Qvb, onde Qv = I −βvv∗ é o refletor de Householder dado por v e β.

In [ ]:
function apply_reflector(v, beta, b)
    n_v = length(v)
    n_b = length(b)

    k = n_b - n_v + 1
    b_view = @view b[k:end]

    alpha = beta .* dot(v, b_view)
    b_view .-= alpha * v
    
    return b
end

apply_reflector (generic function with 1 method)

In [86]:
b = rand(10)
display(b)
r = reflector(b)
apply_reflector(r.v, r.beta, b)

10-element Vector{Float64}:
 0.859126432972095
 0.518591434241383
 0.6677632137038251
 0.6112378703297202
 0.23856534925240325
 0.5495703950947043
 0.29763129095719265
 0.5261424453009946
 0.9187878020262542
 0.7836227237379536

10-element Vector{Float64}:
 2.0022841078739333
 1.1102230246251565e-16
 1.1102230246251565e-16
 1.1102230246251565e-16
 5.551115123125783e-17
 1.1102230246251565e-16
 5.551115123125783e-17
 1.1102230246251565e-16
 2.220446049250313e-16
 2.220446049250313e-16

b) Testes. Verifique que sua função está correta, aplicando em vetores x de mesma dimen
são que v, e depois para vetores de dimensões maiores do que v (adaptando, se necessário,
sua função para funcionar neste caso). Certifique-se que sua função tem complexidade
O(n), onde n é a dimensão do vetor de entrada

c) Generalize sua função para apply_reflector(v, beta, A) que calcula QvA para uma
matriz A (com no mínimo o mesmo número de linhas do que v).

In [92]:
function apply_reflector(v, beta, A)
    n_v = length(v)
    m, n = size(A)

    if m >= n_v
        k = m - n_v + 1

        A_view = @view A[k:end,:]
        vA = v'*A_view
        beta_v = beta .* v
        A_view .-= beta_v * vA
    
        return A
    end
end

apply_reflector (generic function with 1 method)

In [91]:
function rev_apply_reflector(v, beta, A)
    n_v = length(v)
    m, n = size(A)

    if n >= n_v
        k = n - n_v + 1

        A_view = @view A[:,k:end]
        Av = A_view*v
        beta_v = beta .* v'
        A_view .-=  Av * beta_v
    
        return A
    end
end



rev_apply_reflector (generic function with 1 method)

In [95]:
function to_hessemberg(A; return_Q = false)
    H = copy(A)
    m, n = size(H)
    reflectors = []

    if return_Q
        Q = Matrix{eltype(H)}(I, n, n)
    end

    for j in 1:(n-2)
        x = @view H[(j+1):end,j]

        r = reflector(x)
        push!(reflectors, r)

        apply_reflector(r.v, r.beta, @view H[(j+1):end, j:end])
        rev_apply_reflector(r.v, r.beta, @view H[:, (j+1):end])

        if return_Q
            rev_apply_reflector(r.v, r.beta, @view Q[:, (j+1):end])
        end
    end

    if return_Q
        return reflectors, H, Q
    else
        return reflectors, H
    end
end

to_hessemberg (generic function with 1 method)